# Musical Ciphers: The B-A-C-H Motif

## Introduction

A **musical cipher** encodes letters or words as musical notes. Composers have used ciphers to sign their works, pay tribute to colleagues, or hide secret messages in plain hearing.

One of the most famous examples is Bach's own name, encoded as four notes using the **German note-naming system**. Johann Sebastian Bach wove this motif into the final fugue of *The Art of Fugue* (BWV 1080, Contrapunctus XIV) — the manuscript breaks off, possibly at the very moment he wrote his own name into the music.

---

## The German Note-Naming System

Most countries name notes A B C D E F G. Germany uses the same letters, but with one crucial difference:

| German letter | Note |
|:---:|:---:|
| A | A |
| B | B♭ (B-flat) |
| C | C |
| D | D |
| E | E |
| F | F |
| G | G |
| **H** | **B♮ (B-natural)** |

So the letters **B–A–C–H** map directly to the notes **B♭ – A – C – B♮**.

This is not a coincidence — Bach was almost certainly aware of it, and exploited it deliberately.

---
## Part 1: Encoding and Decoding

Let's represent each German letter as a MIDI note number. MIDI is a standard way of representing notes as integers:
- Middle C = 60, C# = 61, D = 62 … and so on.

We'll use octave 4 (the middle octave) throughout.

In [ ]:
# The German cipher: letter -> MIDI note number (octave 4)
# MIDI note 60 = middle C (C4)

GERMAN_CIPHER = {
    'A': 69,  # A4
    'B': 70,  # Bb4  (B-flat)
    'C': 60,  # C4
    'D': 62,  # D4
    'E': 64,  # E4
    'F': 65,  # F4
    'G': 67,  # G4
    'H': 71,  # B4   (B-natural)
}

# Reverse mapping: MIDI note -> letter
MIDI_TO_LETTER = {v: k for k, v in GERMAN_CIPHER.items()}

# Human-readable note names for display
NOTE_NAMES = {
    60: 'C',  61: 'C#', 62: 'D',  63: 'Eb',
    64: 'E',  65: 'F',  66: 'F#', 67: 'G',
    68: 'Ab', 69: 'A',  70: 'Bb', 71: 'B',
}

def encode(text):
    """Encode a string of letters (A-H only) as a list of MIDI note numbers."""
    text = text.upper()
    result = []
    for ch in text:
        if ch in GERMAN_CIPHER:
            result.append(GERMAN_CIPHER[ch])
        elif ch == ' ':
            result.append(None)  # Rest
        else:
            raise ValueError(f"Letter '{ch}' is not in the German cipher (A–H only)")
    return result

def decode(midi_notes):
    """Decode a list of MIDI note numbers back to letters."""
    result = []
    for note in midi_notes:
        if note is None:
            result.append(' ')
        elif note in MIDI_TO_LETTER:
            result.append(MIDI_TO_LETTER[note])
        else:
            result.append('?')
    return ''.join(result)

def describe(midi_notes):
    """Print a human-readable description of encoded notes."""
    for note in midi_notes:
        if note is None:
            print("  REST")
        else:
            letter = MIDI_TO_LETTER.get(note, '?')
            name   = NOTE_NAMES.get(note % 12, '?')
            print(f"  Letter '{letter}'  →  {name}  (MIDI {note})")

print("Cipher ready.")

In [ ]:
# Encode BACH
motif = encode('BACH')
print("B-A-C-H encoded as MIDI notes:")
describe(motif)
print()
print("Decoded back:", decode(motif))

### What does B-A-C-H sound like?

The four notes are: **B♭ – A – C – B♮**

In Western music notation, written out in treble clef, this forms a compact, memorable four-note cell. Bach used it as a *subject* for a fugue — a theme that can be inverted, retrograded, and combined with itself.

Let's also look at two other famous ciphered names using the same system:

In [ ]:
# Other famous names encodable in the German system
for name in ['CAGE', 'BACH', 'BEAD']:
    notes = encode(name)
    note_names = [NOTE_NAMES[n % 12] for n in notes]
    print(f"{name:6s}  →  {' – '.join(note_names)}")

---
## Part 2: Extending the Cipher

The German system only covers 8 letters (A–H). To encode the full alphabet, composers and cryptographers invented extensions. One common approach assigns each letter a note by its position in the alphabet, mapping 26 letters onto the chromatic scale (12 notes) by wrapping around — called a **modular mapping**.

In [ ]:
import string

def make_chromatic_cipher(root_midi=60):
    """
    Map A-Z onto chromatic scale notes starting at root_midi.
    Each letter maps to root + (position mod 12), in octaves as needed.
    """
    cipher = {}
    for i, letter in enumerate(string.ascii_uppercase):
        octave_offset = (i // 12) * 12
        note = root_midi + (i % 12) + octave_offset
        cipher[letter] = note
    return cipher

CHROMATIC_CIPHER = make_chromatic_cipher(root_midi=60)

print("Full A-Z chromatic cipher (starting at C4 = MIDI 60):")
print()
for i, (letter, note) in enumerate(CHROMATIC_CIPHER.items()):
    name = NOTE_NAMES[note % 12]
    octave = (note // 12) - 1
    print(f"  {letter} → {name}{octave} (MIDI {note})", end="    ")
    if (i + 1) % 4 == 0:
        print()

In [ ]:
def encode_chromatic(text):
    """Encode any text using the full chromatic cipher."""
    result = []
    for ch in text.upper():
        if ch in CHROMATIC_CIPHER:
            result.append(CHROMATIC_CIPHER[ch])
        elif ch == ' ':
            result.append(None)
        # Ignore punctuation etc.
    return result

# Try encoding a short phrase
phrase = "HELLO"
encoded = encode_chromatic(phrase)
note_names = [NOTE_NAMES[n % 12] if n else 'REST' for n in encoded]
print(f"'{phrase}'  →  {' – '.join(note_names)}")
print(f"MIDI values: {encoded}")

---
## Part 3: Playing the Cipher with MIDI

We can write an encoded message to a real MIDI file, which can be opened in any notation software or DAW (e.g. GarageBand, MuseScore, Logic).

We'll use the `midiutil` library.

In [ ]:
# Install midiutil if needed
%pip install midiutil --quiet

In [ ]:
from midiutil import MIDIFile

def write_midi(midi_notes, filename, tempo=120, note_duration=1.0, volume=100):
    """
    Write a sequence of MIDI note numbers to a .mid file.
    None values are treated as rests.
    """
    midi = MIDIFile(1)          # 1 track
    midi.addTempo(0, 0, tempo)  # track, time, bpm

    time = 0
    for note in midi_notes:
        if note is not None:
            midi.addNote(
                track=0, channel=0,
                pitch=note,
                time=time,
                duration=note_duration,
                volume=volume
            )
        time += note_duration

    with open(filename, 'wb') as f:
        midi.writeFile(f)
    print(f"Written: {filename}")

# Write the B-A-C-H motif
bach_notes = encode('BACH')
write_midi(bach_notes, 'bach_motif.mid', tempo=60)

# Write it repeated four times, as Bach might use it in a fugue subject
write_midi(bach_notes * 4, 'bach_motif_repeated.mid', tempo=80)

---
## Part 4: Searching for the Motif in a MIDI File

Now for the interesting computational question: **if we have a MIDI file of a piece of music, can we search it for a hidden cipher?**

We have included a sample file — `sample_bach_melody.mid` — a short Bach-style melody with the B-A-C-H motif deliberately embedded at three positions. Let's search it and see if we can find them.

> **How the sample was made:** `generate_sample_midi.py` (in this repo) built the melody programmatically. The motif **B♭–A–C–B♮** appears at beats 0, 8, and 20, surrounded by conjunct stepwise passages so it sounds like a real melodic line rather than a repeated pattern.

Let's write a search function and apply it to the sample:

In [ ]:
%pip install pretty_midi --quiet

In [ ]:
import pretty_midi

def extract_note_sequence(midi_file):
    """
    Extract the sequence of MIDI pitch numbers from a MIDI file,
    sorted by onset time. Returns a flat list of pitch numbers.
    """
    pm = pretty_midi.PrettyMIDI(midi_file)
    notes = []
    for instrument in pm.instruments:
        if not instrument.is_drum:
            for note in instrument.notes:
                notes.append((note.start, note.pitch))
    notes.sort()  # sort by onset time
    return [pitch for (_, pitch) in notes]

def search_motif(note_sequence, motif, label="motif"):
    """
    Search for a motif (list of pitch numbers) within a note sequence.
    Matches pitch class only (ignores octave), so Bb in any octave matches.
    Returns list of positions where the motif starts.
    """
    motif_pc = [n % 12 for n in motif]   # pitch class (0-11)
    seq_pc   = [n % 12 for n in note_sequence]
    n = len(motif_pc)
    matches = []

    for i in range(len(seq_pc) - n + 1):
        if seq_pc[i:i+n] == motif_pc:
            matches.append(i)

    print(f"Searching for {label}: {[NOTE_NAMES[p] for p in motif]}")
    print(f"Found {len(matches)} occurrence(s) at note positions: {matches}")
    return matches

# Search the sample MIDI file for the B-A-C-H motif
# (The motif was embedded at beats 0, 8, and 20 — we expect 3 matches)
seq = extract_note_sequence('sample_bach_melody.mid')
print(f"Full note sequence ({len(seq)} notes):")
print(f"  {[NOTE_NAMES[n % 12] for n in seq]}")
print()
matches = search_motif(seq, encode('BACH'), label='BACH')
print()
print("Expected: 3 matches  ✓" if len(matches) == 3 else f"Got {len(matches)} matches")

---
## Part 5: The Hard Problem — Searching for Unknown Ciphers

The search above works because we **knew** the motif we were looking for. But what if we want to search a piece of music to see whether *any* hidden text is encoded?

This is where computational complexity becomes interesting.

### The problem

Given a piece with $N$ notes and an alphabet of 26 letters:

1. We need to **choose a mapping** from letters → notes. How many mappings are there?
2. For each mapping, we **search** the note sequence for any meaningful text.

### Counting the mappings

If we assign each of 26 letters to one of 12 pitch classes (notes), allowing repeats, the number of possible mappings is:

$$12^{26} \approx 1.7 \times 10^{28}$$

That's more than ten trillion trillion. Let's put that in perspective:

In [ ]:
# How many possible letter -> pitch-class mappings are there?

notes_available = 12   # pitch classes (chromatic scale)
letters = 26           # A-Z

# Each letter independently maps to one of 12 notes
mappings_with_repeats = notes_available ** letters
print(f"Mappings allowing repeats:  {mappings_with_repeats:.2e}")

# If we require each note is used at most once (injective mapping)
# 26 letters but only 12 notes → impossible to be injective!
# So some letters MUST share a note (by pigeonhole principle)

# More constrained: 12 notes across 26 letters, ordered assignment
# (e.g. notes repeat cyclically)
import math
# Permutations of 12 notes taken 12 at a time * ways to partition 26 letters into 12 groups
# is very complex; the simple upper bound is 12^26

print()
print("For context:")
print(f"  Atoms in the observable universe: ~10^80")
print(f"  Seconds since the Big Bang:        ~4 x 10^17")
print(f"  Our search space (12^26):           ~{mappings_with_repeats:.1e}")
print()

# If a computer checks 1 billion mappings per second...
checks_per_second = 1e9
seconds = mappings_with_repeats / checks_per_second
years = seconds / (60 * 60 * 24 * 365.25)
print(f"At 10^9 checks/second, brute force would take: {years:.2e} years")

### Can we do better?

Yes — but it requires **constraints and heuristics**:

- **Known cipher systems** (German, solfège, etc.) reduce the space dramatically
- **Frequency analysis**: in English, E is the most common letter; we might expect the most common note to map to E
- **Language models**: we only need to check if the decoded text looks like real words — modern language models can score this efficiently
- **Musical constraints**: ciphers tend to use a small range of notes and simple rhythms

This is a classic example of a problem that is **computationally intractable** in the general case, but tractable with domain knowledge — a theme throughout computational humanities.

In [ ]:
# A smarter approach: frequency-based cipher recovery
# (simplified demonstration)

from collections import Counter

# English letter frequencies (approximate, most to least common)
ENGLISH_FREQ_ORDER = 'ETAOINSHRDLCUMWFGYPBVKJXQZ'

def frequency_attack(note_sequence, top_n=5):
    """
    Attempt to recover a cipher by mapping the most frequent notes
    to the most frequent English letters.
    Returns a candidate cipher mapping.
    """
    # Count pitch class frequencies
    pc_seq = [n % 12 for n in note_sequence]
    freq = Counter(pc_seq)
    most_common_notes = [pc for pc, _ in freq.most_common()]

    print("Note frequencies (most common first):")
    for pc, count in freq.most_common():
        print(f"  {NOTE_NAMES[pc]:3s}  {count:3d} times")

    print()
    print("Candidate cipher (frequency mapping):")
    candidate = {}
    for i, pc in enumerate(most_common_notes):
        if i < len(ENGLISH_FREQ_ORDER):
            candidate[ENGLISH_FREQ_ORDER[i]] = pc
            print(f"  {ENGLISH_FREQ_ORDER[i]} → {NOTE_NAMES[pc]}")

    return candidate

# Demonstrate on the sample melody
seq = extract_note_sequence('sample_bach_melody.mid')
candidate = frequency_attack(seq)

---
## Summary

| Concept | What we did |
|---|---|
| **German cipher** | Mapped A–H to notes, encoded and decoded BACH |
| **Chromatic extension** | Extended to full A–Z using the 12-note scale |
| **MIDI output** | Wrote encoded text to a playable `.mid` file |
| **Motif search** | Searched a note sequence for a known target |
| **Complexity** | Showed brute-force search is $O(12^{26})$ — intractable |
| **Frequency analysis** | A smarter heuristic approach to cipher recovery |

### Questions for discussion

1. Bach's cipher only works because of the **German naming convention**. What does this tell us about how cultural context shapes what counts as a "hidden" message?
2. The B-A-C-H motif appears in works by Schumann, Liszt, and many others as a homage. Is that still a "cipher", or something else?
3. What additional constraints could you add to make the brute-force search tractable?
4. Could a composer today design a cipher that resists both frequency analysis *and* musical convention?

---
*Notebook for the Humanities Python class — Musical Ciphers module*